# 第八章 搭建一个带评估的端到端问答系统

 - [一、环境配置](#一、环境配置)
 - [二、用于处理用户查询的链式 Prompt 系统](#二、用于处理用户查询的链式-Prompt-系统)
     - [2.1 一个端到端实现问答的函数](#2.1-一个端到端实现问答的函数)
     - [2.2 持续收集用户和助手消息的函数](#2.2-持续收集用户和助手消息的函数)


在本章中，我们将搭建一个带评估的端到端问答系统，这个系统综合了之前多节课的内容，并加入了评估过程。

1. 检查输入，确认其是否能通过审核 API 的审核。

2. 如果通过了审核，我们将查找产品列表。

3. 如果找到了产品，我们将尝试查找它们的相关信息。

4. 我们使用模型回答用户提出的问题。

5. 我们将通过审核 API 对生成的答案进行审核。

如果没有被标记为有害的，我们将把答案返回给用户。

## 一、环境配置

同上一章，我们首先需要配置使用 OpenAI API 的环境

In [1]:
# 配置 OpenAI KEY
import os
import sys
from openai import OpenAI
from dotenv import load_dotenv, find_dotenv
from IPython.display import Markdown
sys.path.append('../..')
# 使用英文 Prompt 的工具包
import utils_en
# 使用中文 Prompt 的工具包
import utils_zh

import panel as pn  # 用于图形化界面
pn.extension()


loaded = load_dotenv(find_dotenv(), override=True)
# 从环境变量中获取 OpenAI API Key 或者直接赋值
API_KEY = os.getenv("API_KEY")


# 如果您使用的是官方 API，就直接用 https://api.siliconflow.cn/v1 就行。
BASE_URL = "https://api.siliconflow.cn/v1"

In [2]:
# 实例化 OpenAI 对象
# 传入参数：OpenAI API Key（必需）、Base URL 和最大重试次数
client = OpenAI(api_key=API_KEY, base_url=BASE_URL, max_retries=3)

In [3]:
def get_completion_from_messages(messages, 
                                 model_endpoint, 
                                 temperature=0, 
                                 max_tokens=500):
    '''
    封装一个支持更多参数的自定义访问 OpenAI GPT3.5 的函数

    参数: 
    messages: 这是一个消息列表，每个消息都是一个字典，包含 role(角色）和 content(内容)。角色可以是'system'、'user' 或 'assistant’，内容是角色的消息。
    model: 调用的模型，默认为 gpt-3.5-turbo(ChatGPT)，有内测资格的用户可以选择 gpt-4
    temperature: 这决定模型输出的随机程度，默认为0，表示输出将非常确定。增加温度会使输出更随机。
    max_tokens: 这决定模型输出的最大的 token 数。
    '''
    extra_body = {}
    if "Qwen3" in model_endpoint:
        extra_body={
            "enable_thinking": False
        }
        
    response = client.chat.completions.create(model=model_endpoint,
                                              messages=messages,
                                              n=1, temperature=temperature, seed=42,
                                              presence_penalty=0, frequency_penalty=0,
                                              max_tokens=max_tokens, extra_body = extra_body
                                             )
    return response.choices[0].message.content.strip()

## 二、用于处理用户查询的链式 Prompt 系统

### 2.1 一个端到端实现问答的函数

In [4]:
def process_user_message(user_input, model_endpoint, all_messages, debug=True):
    """
    对用户信息进行预处理
    
    参数:
    user_input : 用户输入
    all_messages : 历史信息
    debug : 是否开启 DEBUG 模式,默认开启
    """
    # 分隔符
    delimiter = "```"
    
    # 第一步: 使用 OpenAI 的 Moderation API 检查用户输入是否合规或者是一个注入的 Prompt
    # response = openai.Moderation.create(input=user_input)
    # moderation_output = response["results"][0]

    # # 经过 Moderation API 检查该输入不合规
    # if moderation_output["flagged"]:
    #     print("第一步：输入被 Moderation 拒绝")
    #     return "抱歉，您的请求不合规"

    # # 如果开启了 DEBUG 模式，打印实时进度
    if debug: print("第一步：输入通过 Moderation 检查")
    
    # 第二步：抽取出商品和对应的目录，类似于之前课程中的方法，做了一个封装
    category_and_product_response = utils_en.find_category_and_product_only_model(user_input, model_endpoint, utils_en.get_products_and_category())
    #print(category_and_product_response)
    # 将抽取出来的字符串转化为列表
    category_and_product_list = utils_en.read_string_to_list(category_and_product_response)
    #print(category_and_product_list)

    if debug: print("第二步：抽取出商品列表")

    # 第三步：查找商品对应信息
    product_information = utils_en.generate_output_string(category_and_product_list)
    if debug: print("第三步：查找抽取出的商品信息")

    # 第四步：根据信息生成回答
    system_message = f"""
    You are a customer service assistant for a large electronic store. \
    Respond in a friendly and helpful tone, with concise answers. \
    Make sure to ask the user relevant follow-up questions.
    """
    # 插入 message
    messages = [
        {'role': 'system', 'content': system_message},

        # ⬇️ 1) 把 JSON 当成“参考资料”放到 system 里
        {'role': 'system',  # 这里改成 system
        'content': f"Relevant product data (reference only):\n{product_information}\n\n"
                    "When replying, respond in friendly natural language, summarise key specs, "
                    "and ask follow‑up questions. Do NOT output JSON."},

        # ⬇️ 2) 再放用户真正的问题
        {'role': 'user', 'content': f"{delimiter}{user_input}{delimiter}"}
    ]
    # 获取 GPT3.5 的回答
    # 通过附加 all_messages 实现多轮对话
    final_response = get_completion_from_messages(all_messages + messages, model_endpoint)
    if debug:print("第四步：生成用户回答")
    # 将该轮信息加入到历史信息中
    all_messages = all_messages + messages[1:]

    # # 第五步：基于 Moderation API 检查输出是否合规
    # response = openai.Moderation.create(input=final_response)
    # moderation_output = response["results"][0]

    # # 输出不合规
    # if moderation_output["flagged"]:
    #     if debug: print("第五步：输出被 Moderation 拒绝")
    #     return "抱歉，我们不能提供该信息"

    if debug: print("第五步：输出经过 Moderation 检查")

    # 第六步：模型检查是否很好地回答了用户问题
    user_message = f"""
    Customer message: {delimiter}{user_input}{delimiter}
    Agent response: {delimiter}{final_response}{delimiter}

    Does the assistant response fully and clearly answer the question?
    Reply with one letter only: "Y" for Yes, or "N" for No.
    """

    messages = [
        {'role': 'system', 'content': "You are an evaluator for customer service conversations."},
        {'role': 'user', 'content': user_message}
    ]

    # temperature 设为 0，让模型不发散
    evaluation_response = get_completion_from_messages(messages, model_endpoint, temperature=0)
    if debug: print("第六步：模型评估该回答")

    # 第七步：如果评估为 Y，输出回答；如果评估为 N，反馈将由人工修正答案
    if "Y" in evaluation_response.upper():  # 使用 in 来避免模型可能生成 Yes
        if debug: print("第七步：模型赞同了该回答.")
        return final_response, all_messages
    else:
        if debug: print("第七步：模型不赞成该回答.")
        neg_str = "很抱歉，我无法提供您所需的信息。我将为您转接到一位人工客服代表以获取进一步帮助。"
        return neg_str, all_messages

user_input = "tell me about the smartx pro phone and the fotosnap camera, the dslr one. Also what tell me about your tvs"
response,_ = process_user_message(user_input, "Qwen/Qwen3-8B", [])
print(response)

第一步：输入通过 Moderation 检查
第二步：抽取出商品列表
第三步：查找抽取出的商品信息
第四步：生成用户回答
第五步：输出经过 Moderation 检查
第六步：模型评估该回答
第七步：模型赞同了该回答.
The **SmartX ProPhone (SX-PP10)** is a powerful smartphone with a 6.1-inch display, 128GB storage, a 12MP dual camera, and 5G connectivity. It's great for everyday use and capturing high-quality photos on the go.  

The **FotoSnap DSLR Camera (FS-DSLR200)** is a versatile camera with a 24.2MP sensor, 1080p video recording, a 3-inch LCD screen, and interchangeable lenses. It's perfect for photographers who want more control and better image quality.  

As for our TVs, we offer a range of models, including the **CineView 4K TV (CV-4K55)**, **CineView 8K TV (CV-8K65)**, and **CineView OLED TV (CV-OLED55)**. These TVs provide stunning visuals with 4K, 8K, and OLED technology, respectively, along with HDR support and smart features.  

Would you like more details about any of these products or help choosing the best one for your needs?


In [14]:
'''
注意：限于模型对中文理解能力较弱，中文 Prompt 可能会随机出现不成功，可以多次运行；也非常欢迎同学探究更稳定的中文 Prompt
'''
def process_user_message_ch(user_input, model_endpoint, all_messages, debug=True):
    """
    对用户信息进行预处理
    
    参数:
    user_input : 用户输入
    all_messages : 历史信息
    debug : 是否开启 DEBUG 模式,默认开启
    """
    # 分隔符
    delimiter = "```"
    
    # 第一步: 使用 OpenAI 的 Moderation API 检查用户输入是否合规或者是一个注入的 Prompt
    # response = openai.Moderation.create(input=user_input)
    # moderation_output = response["results"][0]

    # # 经过 Moderation API 检查该输入不合规
    # if moderation_output["flagged"]:
    #     print("第一步：输入被 Moderation 拒绝")
    #     return "抱歉，您的请求不合规"

    # 如果开启了 DEBUG 模式，打印实时进度
    if debug: print("第一步：输入通过 Moderation 检查")
    
    # 第二步：抽取出商品和对应的目录，类似于之前课程中的方法，做了一个封装
    category_and_product_response = utils_zh.find_category_and_product_only_model(user_input, model_endpoint, utils_zh.get_products_and_category())
    # 将抽取出来的字符串转化为列表
    category_and_product_list = utils_zh.read_string_to_list(category_and_product_response)

    if debug: print("第二步：抽取出商品列表")

    # 第三步：查找商品对应信息
    product_information = utils_zh.generate_output_string(category_and_product_list)
    if debug: print("第三步：查找抽取出的商品信息")

    # 第四步：根据信息生成回答
    system_message = (
        "您是大型电子产品电商的客服人员，任务是用友好、简洁的语气回答用户问题，"
        "可使用参考资料，但不输出原始数据结构，只提供自然语言摘要。"
    )
    # 插入 message
    messages = [
            # 客服角色约束
            {"role": "system", "content": system_message},

            # 参考资料（JSON）
            {
                "role": "system",
                "content": (
                    f"以下是参考商品数据（仅供内部参考，请勿直接输出 JSON）：\n"
                    f"{product_information}"
                )
            },

            # 用户真实提问
            {"role": "user", "content": f"{delimiter}{user_input}{delimiter}"},
        ]
    # 获取 GPT3.5 的回答
    # 通过附加 all_messages 实现多轮对话
    final_response = get_completion_from_messages(all_messages + messages, model_endpoint)
    
    if debug:print("第四步：生成用户回答")
    # 将该轮信息加入到历史信息中
    all_messages = all_messages + messages[1:]

    # 第五步：基于 Moderation API 检查输出是否合规
    # response = openai.Moderation.create(input=final_response)
    # moderation_output = response["results"][0]

    # # 输出不合规
    # if moderation_output["flagged"]:
    #     if debug: print("第五步：输出被 Moderation 拒绝")
    #     return "抱歉，我们不能提供该信息"

    if debug: print("第五步：输出经过 Moderation 检查")

    # 第六步：模型检查是否很好地回答了用户问题
    eval_prompt = (
            f"用户问题：{delimiter}{user_input}{delimiter}\n"
            f"客服回复：{delimiter}{final_response}{delimiter}\n\n"
            "请评估该回复是否完整清晰地解答了用户问题。\n"
            "若是，请仅输出字母 Y；若否，仅输出字母 N。"
    )

    eval_messages = [
        {"role": "system", "content": "你是客服对话评估员，只能回答 Y 或 N。"},
        {"role": "user", "content": eval_prompt},
    ]

    evaluation_response = get_completion_from_messages(eval_messages,
                                                       model_endpoint,
                                                       temperature=0)
    if debug: print("第六步：模型评估该回答")

    # 第七步：如果评估为 Y，输出回答；如果评估为 N，反馈将由人工修正答案
    if "Y" in evaluation_response:  # 使用 in 来避免模型可能生成 Yes
        if debug: print("第七步：模型赞同了该回答.")
        return final_response, all_messages
    else:
        if debug: print("第七步：模型不赞成该回答.")
        neg_str = "很抱歉，我无法提供您所需的信息。我将为您转接到一位人工客服代表以获取进一步帮助。"
        return neg_str, all_messages

user_input = "请告诉我关于 smartx pro phone 和 the fotosnap camera 的信息。另外，请告诉我关于你们的tvs的情况。"
response,_ = process_user_message_ch(user_input,"Qwen/Qwen3-8B", [])
print(response)

第一步：输入通过 Moderation 检查
第二步：抽取出商品列表
第三步：查找抽取出的商品信息
第四步：生成用户回答
第五步：输出经过 Moderation 检查
第六步：模型评估该回答
第七步：模型赞同了该回答.
以下是您询问的产品信息：

**SmartX ProPhone**  
这是一款功能强大的智能手机，配备6.1英寸显示屏、128GB存储空间、12MP双摄像头以及支持5G网络。它拥有4.6的高评分，价格为899.99元。

**FotoSnap 相机系列**  
- **FotoSnap DSLR Camera**：具备24.2MP传感器、1080p视频拍摄、3英寸LCD屏幕和可更换镜头，适合追求专业摄影体验的用户，价格为599.99元。  
- **FotoSnap Mirrorless Camera**：轻巧便携，拥有20.1MP传感器、4K视频拍摄、3英寸触摸屏和可更换镜头，价格为799.99元。  
- **FotoSnap Instant Camera**：一款便携式即时打印相机，支持直闪、自拍镜和电池供电，价格为69.99元。

**关于我们的电视产品**  
我们有多种电视可供选择，包括：  
- **CineView 4K TV**：55英寸4K分辨率电视，支持HDR和智能功能，价格为599.99元，提供2年保修。  
- **CineView 8K TV**：65英寸8K超高清电视，拥有HDR和智能功能，价格为2999.99元，提供2年保修。  
- **CineView OLED TV**：55英寸OLED屏幕，提供出色的黑色表现和色彩，支持4K分辨率和HDR，价格为1499.99元，提供2年保修。  

如需了解更多详情或帮助选购，请随时告诉我！


### 2.2 持续收集用户和助手消息的函数

实现一个可视化界面

In [15]:
def collect_messages_en(debug=False):
    """
    用于收集用户的输入并生成助手的回答

    参数：
    debug: 用于觉得是否开启调试模式
    """
    user_input = inp.value_input
    if debug: print(f"User Input = {user_input}")
    if user_input == "":
        return
    inp.value = ''
    global context
    # 调用 process_user_message 函数
    #response, context = process_user_message(user_input, context, utils.get_products_and_category(),debug=True)
    response, context = process_user_message(user_input, context, debug=False)
    context.append({'role':'assistant', 'content':f"{response}"})
    panels.append(
        pn.Row('User:', pn.pane.Markdown(user_input, width=600)))
    panels.append(
        pn.Row('Assistant:', pn.pane.Markdown(response, width=600, style={'background-color': '#F6F6F6'})))
 
    return pn.Column(*panels) # 包含了所有的对话信息

In [20]:
panels = [] # collect display 

# 系统信息
context = [ {'role':'system', 'content':"You are Service Assistant"} ]  

inp = pn.widgets.TextInput( placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="Service Assistant")

interactive_conversation = pn.bind(collect_messages_en, button_conversation)

dashboard = pn.Column(
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=300),
)

dashboard

e:\Anaconda\envs\python3.12\Lib\site-packages\panel\viewable.py:298: ParamFutureWarning: Parameter 'object' on <class 'panel.pane.base.PaneBase'> is being given a valid parameter reference <function _param_bind.<locals>.wrapped at 0x000001FEA810F560> but is implicitly allow_refs=False. In future allow_refs will be enabled by default and the reference <function _param_bind.<locals>.wrapped at 0x000001FEA810F560> will be resolved to its underlying value None. Please explicitly set allow_ref on the Parameter definition to declare whether references should be resolved or not.
  super().__init__(**params)


BokehModel(combine_events=True, render_bundle={'docs_json': {'77c0a013-03d1-4eb7-bb73-7cd38a9dced0': {'version…

In [ ]:
# 调用中文 Prompt 版本
def collect_messages_ch(debug=False):
    """
    用于收集用户的输入并生成助手的回答

    参数：
    debug: 用于觉得是否开启调试模式
    """
    user_input = inp.value_input
    if debug: print(f"User Input = {user_input}")
    if user_input == "":
        return
    inp.value = ''
    global context
    # 调用 process_user_message 函数
    #response, context = process_user_message(user_input, context, utils.get_products_and_category(),debug=True)
    response, context = process_user_message_ch(user_input, context, debug=False)
    context.append({'role':'assistant', 'content':f"{response}"})
    panels.append(
        pn.Row('User:', pn.pane.Markdown(user_input, width=600)))
    panels.append(
        pn.Row('Assistant:', pn.pane.Markdown(response, width=600, style={'background-color': '#F6F6F6'})))
 
    return pn.Column(*panels) # 包含了所有的对话信息

In [ ]:
panels = [] # collect display 

# 系统信息
context = [ {'role':'system', 'content':"You are Service Assistant"} ]  

inp = pn.widgets.TextInput( placeholder='Enter text here…')
button_conversation = pn.widgets.Button(name="Service Assistant")

interactive_conversation = pn.bind(collect_messages_ch, button_conversation)

dashboard = pn.Column(
    inp,
    pn.Row(button_conversation),
    pn.panel(interactive_conversation, loading_indicator=True, height=300),
)

dashboard

通过监控系统在更多输入上的质量，您可以修改步骤，提高系统的整体性能。

也许我们会发现，对于某些步骤，我们的提示可能更好，也许有些步骤甚至不必要，也许我们会找到更好的检索方法等等。

我们将在下一章中进一步讨论这个问题。 